# 27. Embedding과 벡터 검색

> **제27장** · **이론편 대응: 20.3절(Embedding), 21.2~21.3절(RAG의 검색)**
> **예상 소요**: 70분
> **필요 사양**: **[CPU]** 로 실행 가능
> **추가 설치**: **sentence-transformers** (1절 참조)
> **다운로드**: 임베딩 모델 약 470MB (자동)

---

## 이 장에서 하는 일

지금까지는 텍스트를 **생성**했다. 이번에는 텍스트를 **찾는** 문제를 다룬다.

| 절 | 하는 일 | 이론편 대응 |
|---|---|---|
| 1 | **준비 — sentence-transformers 설치** | — |
| 2 | **코사인 유사도 손계산 검증** ★ | 21.3절 |
| 3 | 실제 임베딩 모델 사용 | 20.3절 |
| 4 | **키워드 검색과 무엇이 다른가** | 21.2절 |
| 5 | 모델 선택 — 영어 전용 vs 다국어 | — |
| 6 | 벡터 검색 구현 | 21.3절 |
| 7 | 문서 분할(Chunking) | 21.2절 |
| 8 | 검색 품질 평가 | 21.5절 |

**이 장이 23번 RAG의 앞부분**이다. 검색이 제대로 되지 않으면
아무리 좋은 LLM을 붙여도 소용없기 때문에, 검색 자체를 먼저 확실히 해 둔다.

---

## 1. 준비 — sentence-transformers 설치

### 왜 별도 라이브러리인가

23장에서 쓴 `transformers`로도 임베딩을 만들 수 있다. 하지만 몇 가지 처리가 더 필요하다.

| 필요한 처리 | 직접 하면 | sentence-transformers |
|---|---|---|
| 토큰별 벡터 → 문장 벡터 | 평균 내기(pooling) 직접 구현 | 자동 |
| 정규화 | 직접 | 옵션 하나 |
| 배치 처리 | 직접 | 자동 |
| 유사도 계산 | 직접 | 함수 제공 |

**문장 단위 임베딩에 특화된 라이브러리**라고 보면 된다.

### 설치

```
pip install sentence-transformers
```

`transformers`와 `torch`가 이미 있으면 금방 설치된다.

### 이 장에서 쓸 모델

| 모델 | 용도 | 크기 |
|---|---|---|
| `all-MiniLM-L6-v2` | 영어 전용, 빠름 | 약 90MB |
| `paraphrase-multilingual-MiniLM-L12-v2` | **다국어(한국어 포함)** | 약 470MB |

5절에서 두 모델을 비교하며 **왜 다국어 모델이 필요한지** 확인한다.

In [ ]:
import importlib

print("=" * 60)
print("필요 패키지 확인")
print("=" * 60)

required = [
    ("sentence_transformers", "문장 임베딩", "pip install sentence-transformers"),
    ("transformers", "기반 라이브러리", "pip install transformers"),
    ("torch", "PyTorch", "pip install torch --index-url https://download.pytorch.org/whl/cu128"),
    ("numpy", "수치 계산", "pip install numpy"),
]

missing = []
for name, desc, install in required:
    try:
        mod = importlib.import_module(name)
        ver = getattr(mod, "__version__", "설치됨")
        print(f"[OK]   {name:<24}{ver:<12}{desc}")
    except ImportError:
        print(f"[없음] {name:<24}{'':<12}{desc}")
        missing.append(install)

print("-" * 60)
if missing:
    print("설치가 필요합니다:")
    for cmd in set(missing):
        print(f"  {cmd}")
    print()
    print("설치 후 커널을 재시작하세요.")
else:
    print("[준비 완료] 2절로 진행하세요.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

np.set_printoptions(precision=4, suppress=True)
print("준비 완료")

---

## 2. 코사인 유사도 — 이론편 21.3절 값 검증 ★

이론편 21.3절에서 질문 "강아지 사료"와 세 문서의 유사도를 손으로 계산했다.

**설정**: 질문 벡터가 $(1, 1, 0)$이고, 세 문서가 각각 다음과 같다.

| 문서 | 벡터 | 이론편 결과 |
|---|---|---|
| 강아지 사료 후기 | (1.0, 1.0, 0.0) | **1.000** |
| 반려견 먹이 고르는 법 | (0.9, 0.8, 0.1) | **0.995** |
| 자동차 정비 요령 | (0.1, 0.0, 1.0) | **0.070** |

코사인 유사도의 정의는 이렇다.

$$\cos\theta = \frac{\mathbf{a}\cdot\mathbf{b}}{\lVert\mathbf{a}\rVert\,\lVert\mathbf{b}\rVert}$$

In [ ]:
import numpy as np


def cosine_similarity(a, b):
    """코사인 유사도 (이론편 21.3절)"""
    dot = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    return dot / (norm_a * norm_b)


# 이론편 21.3절과 완전히 같은 값
query = np.array([1.0, 1.0, 0.0])

documents = {
    "강아지 사료 후기":      np.array([1.0, 1.0, 0.0]),
    "반려견 먹이 고르는 법": np.array([0.9, 0.8, 0.1]),
    "자동차 정비 요령":      np.array([0.1, 0.0, 1.0]),
}

print("=" * 70)
print("이론편 21.3절 값 검증")
print("=" * 70)
print(f"질문 벡터: {query}")
print()
print(f"{'문서':<22}{'내적':<10}{'|q|':<10}{'|d|':<10}{'코사인':<10}{'이론편'}")
print("-" * 70)

book_values = [1.000, 0.995, 0.070]
results = {}

for (name, doc), book in zip(documents.items(), book_values):
    dot = np.dot(query, doc)
    nq = np.linalg.norm(query)
    nd = np.linalg.norm(doc)
    cos = cosine_similarity(query, doc)
    results[name] = cos
    print(f"{name:<22}{dot:<10.2f}{nq:<10.3f}{nd:<10.3f}{cos:<10.4f}{book:.3f}")
    assert abs(cos - book) < 0.001, f"{name}이 이론편 값과 다릅니다"

print("-" * 70)
print("[OK] 이론편 21.3절 손계산과 일치")

In [ ]:
import numpy as np

print("=" * 65)
print("두 번째 문서를 손으로 따라가기 (이론편 21.3절)")
print("=" * 65)

q = np.array([1.0, 1.0, 0.0])
d = np.array([0.9, 0.8, 0.1])

print(f"질문 q = {q}")
print(f"문서 d = {d}")
print()

# 1단계: 내적
print("[1단계] 내적")
terms = [f"{q[i]}x{d[i]}" for i in range(3)]
products = [q[i] * d[i] for i in range(3)]
print(f"  q · d = {' + '.join(terms)}")
print(f"        = {' + '.join(f'{p:.2f}' for p in products)}")
print(f"        = {sum(products):.2f}")
print()

# 2단계: 길이
print("[2단계] 각 벡터의 길이")
print(f"  |q| = sqrt({q[0]}^2 + {q[1]}^2 + {q[2]}^2) = sqrt({np.sum(q**2):.2f}) = {np.linalg.norm(q):.4f}")
print(f"  |d| = sqrt({d[0]}^2 + {d[1]}^2 + {d[2]}^2) = sqrt({np.sum(d**2):.2f}) = {np.linalg.norm(d):.4f}")
print()

# 3단계: 나누기
print("[3단계] 나누기")
cos = np.dot(q, d) / (np.linalg.norm(q) * np.linalg.norm(d))
print(f"  {np.dot(q,d):.2f} / ({np.linalg.norm(q):.4f} x {np.linalg.norm(d):.4f})")
print(f"  = {np.dot(q,d):.2f} / {np.linalg.norm(q)*np.linalg.norm(d):.4f}")
print(f"  = {cos:.4f}")
print()
print("-" * 65)
print("이론편 값: 0.995")
assert abs(cos - 0.995) < 0.001
print("[OK] 일치")

### 왜 내적만 쓰지 않고 길이로 나누는가

이론편 21.3절에서 짚은 내용이다. **내적만 보면 벡터가 길기만 해도 값이 커진다.**

같은 방향인데 길이만 두 배인 문서를 만들어 확인해 보자.

In [ ]:
import numpy as np

print("=" * 65)
print("내적 vs 코사인 — 길이의 영향")
print("=" * 65)

q = np.array([1.0, 1.0, 0.0])
base = np.array([0.9, 0.8, 0.1])

print(f"{'문서':<24}{'벡터':<24}{'내적':<12}{'코사인'}")
print("-" * 65)
for mult, label in [(1.0, "원본"), (2.0, "길이 2배"), (0.5, "길이 절반")]:
    d = base * mult
    dot = np.dot(q, d)
    cos = cosine_similarity(q, d)
    print(f"{label:<24}{str(d.round(2)):<24}{dot:<12.3f}{cos:.4f}")

print("-" * 65)
print()
print("내적은 길이에 따라 달라지지만, 코사인은 항상 같다.")
print()
print("왜 중요한가")
print("  긴 문서일수록 벡터가 길어지는 경향이 있다.")
print("  내적만 쓰면 '내용이 관련 있어서'가 아니라 '길어서' 상위에 오를 수 있다.")
print("  코사인은 순수하게 **방향의 일치 정도**만 본다.")
print()
print("직각인 경우 (이론편 4.5절)")
a = np.array([1.0, 0.0])
b = np.array([0.0, 1.0])
print(f"  {a} 와 {b} 의 코사인: {cosine_similarity(a, b):.4f}")
print("  → 0 = 완전히 무관")

---

## 3. 실제 임베딩 모델 — 이론편 20.3절

2절에서는 벡터를 사람이 정해 줬다. 실제로는 **모델이 텍스트를 벡터로 바꾼다.**

이론편 20.3절에서 다룬 Embedding이 바로 이것이다. 의미가 비슷한 텍스트가
가까운 위치에 놓이도록 학습된 변환이다.

In [ ]:
import time
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

print("=" * 60)
print("임베딩 모델 불러오기")
print("=" * 60)
print(f"모델: {MODEL_NAME}")
print("처음 실행하면 약 90MB를 내려받습니다.")
print()

t0 = time.time()
# ── SentenceTransformer 파라미터 ─────────────────────────────
#   model_name_or_path  모델 이름 또는 경로.  **필수**
#   device       실행 장치.  기본값 None(자동 감지)
#                'cuda' / 'cpu' / 'mps'(애플 실리콘)
#   cache_folder 캐시 위치.  기본값 ~/.cache
#   trust_remote_code  기본값 False
#
#   [모델 선택이 성능을 좌우한다]
#     한국어를 다루면 반드시 다국어 모델을 쓴다.
#     영어 전용 모델은 한국어에서 성능이 절반으로 떨어진다.
# ──────────────────────────────────────────────────────────────
model_en = SentenceTransformer(MODEL_NAME)
print(f"로드 완료: {time.time()-t0:.1f}초")
print()

print("모델 정보")
print(f"  임베딩 차원   : {model_en.get_sentence_embedding_dimension()}")
print(f"  최대 입력 길이: {model_en.max_seq_length} 토큰")
n_params = sum(p.numel() for p in model_en.parameters())
print(f"  파라미터      : {n_params/1e6:.1f}M")
print()
print("23장의 GPT-2(124M)와 비교하면 작다.")
print("  생성이 아니라 '벡터로 바꾸기'만 하면 되므로 작아도 된다.")

In [ ]:
import numpy as np

sentences = [
    "The cat sits on the mat.",
    "A kitten is resting on the rug.",
    "I need to fix my car engine.",
]

print("=" * 65)
print("텍스트를 벡터로")
print("=" * 65)

embeddings = model_en.encode(sentences)

print(f"입력: 문장 {len(sentences)}개")
print(f"출력: {embeddings.shape}   ← (문장 수, 차원)")
print()
print("첫 문장의 벡터 (앞 8개 값만)")
print(f"  {embeddings[0][:8].round(4)}")
print(f"  ... 총 {embeddings.shape[1]}개 숫자")
print()

print("문장 사이 유사도")
print(f"{'':6}" + "".join(f"{i:>10}" for i in range(len(sentences))))
print("-" * 65)
for i in range(len(sentences)):
    row = "".join(f"{cosine_similarity(embeddings[i], embeddings[j]):>10.4f}"
                  for j in range(len(sentences)))
    print(f"{i:<6}{row}")
print("-" * 65)
for i, s in enumerate(sentences):
    print(f"  {i}: {s}")
print()

sim_01 = cosine_similarity(embeddings[0], embeddings[1])
sim_02 = cosine_similarity(embeddings[0], embeddings[2])
print(f"0 ↔ 1 (고양이, 같은 뜻 다른 표현): {sim_01:.4f}")
print(f"0 ↔ 2 (고양이 vs 자동차)        : {sim_02:.4f}")
print()
print("겹치는 단어가 하나도 없는데도 0장과 1장이 가깝다.")
print("→ 이론편 20.3절에서 말한 '의미를 담는 벡터'")

### 정규화 — `normalize_embeddings=True`

코사인 유사도를 계산할 때마다 길이로 나누는 것은 번거롭다.
**미리 길이를 1로 맞춰 두면** 내적만으로 코사인 유사도가 된다.

$$\lVert a \rVert = \lVert b \rVert = 1 \;\Rightarrow\; \cos\theta = \mathbf{a}\cdot\mathbf{b}$$

문서가 수만 개일 때 이 차이가 크다. 행렬 곱 한 번으로 전부 계산할 수 있기 때문이다.

In [ ]:
import numpy as np
import time

print("=" * 65)
print("정규화의 효과")
print("=" * 65)

emb_raw = model_en.encode(sentences)
# ── .encode() 파라미터 ───────────────────────────────────────
#   sentences             문자열 또는 리스트.  **필수**
#   batch_size            기본값 32.  GPU 메모리에 맞춰 조절
#   normalize_embeddings  L2 정규화.  기본값 False
#                         **True 로 하면 내적 = 코사인 유사도**
#                         검색에는 반드시 True 를 쓴다
#   convert_to_numpy      기본값 True.  False 면 텐서 반환
#   convert_to_tensor     기본값 False
#   show_progress_bar     기본값 None(자동)
#                         주피터에서는 False 로 두면 깔끔하다
#   output_value          'sentence_embedding'(기본) / 'token_embeddings'
# ──────────────────────────────────────────────────────────────
emb_norm = model_en.encode(sentences, normalize_embeddings=True)

print("정규화 전 벡터 길이")
print(f"  {np.linalg.norm(emb_raw, axis=1).round(4)}")
print("정규화 후 벡터 길이")
print(f"  {np.linalg.norm(emb_norm, axis=1).round(4)}")
print()

# 두 방법이 같은 결과를 주는지
cos_manual = cosine_similarity(emb_raw[0], emb_raw[1])
cos_dot = np.dot(emb_norm[0], emb_norm[1])
print(f"코사인 (직접 계산)        : {cos_manual:.6f}")
print(f"내적 (정규화된 벡터)      : {cos_dot:.6f}")
print(f"같은가: {abs(cos_manual - cos_dot) < 1e-5}")
print()

# 속도 차이 — 문서가 많을 때
print("-" * 65)
print("문서 5000개를 검색할 때 (모의 실험)")
rng = np.random.RandomState(0)
docs = rng.randn(5000, 384).astype(np.float32)
docs_norm = docs / np.linalg.norm(docs, axis=1, keepdims=True)
q = rng.randn(384).astype(np.float32)
q_norm = q / np.linalg.norm(q)

t0 = time.time()
sims_slow = np.array([cosine_similarity(q, d) for d in docs])
t_slow = time.time() - t0

t0 = time.time()
sims_fast = docs_norm @ q_norm
t_fast = time.time() - t0

print(f"  반복문으로 하나씩: {t_slow*1000:8.1f} ms")
print(f"  행렬 곱 한 번    : {t_fast*1000:8.1f} ms")
print(f"  차이            : {t_slow/t_fast:8.0f}배")
print(f"  결과 일치       : {np.allclose(sims_slow, sims_fast, atol=1e-5)}")

---

## 4. 키워드 검색과 무엇이 다른가 — 이론편 21.2절

이론편 21.2절에서 "키워드 검색은 표현이 다르면 못 찾는다"고 했다. 직접 비교해 보자.

**키워드 검색**: 단어가 겹치는지만 본다
**의미 검색**: 벡터 사이 거리를 본다

In [ ]:
import numpy as np
import re


def keyword_search(query, documents):
    """단순 키워드 검색 — 겹치는 단어 수로 점수"""
    q_words = set(re.findall(r"\w+", query.lower()))
    scores = []
    for doc in documents:
        d_words = set(re.findall(r"\w+", doc.lower()))
        overlap = len(q_words & d_words)
        scores.append(overlap / max(len(q_words), 1))
    return np.array(scores)


def semantic_search(query, documents, model):
    """의미 검색 — 임베딩 코사인 유사도"""
    q_emb = model.encode([query], normalize_embeddings=True)[0]
    d_emb = model.encode(documents, normalize_embeddings=True)
    return d_emb @ q_emb


docs_en = [
    "How to train a puppy at home",
    "Best food for young dogs",
    "Car engine maintenance guide",
    "Python programming basics",
    "Feeding schedule for canines",
]

query = "What should I feed my dog?"

print("=" * 70)
print("키워드 검색 vs 의미 검색")
print("=" * 70)
print(f"질문: {query}")
print()

kw_scores = keyword_search(query, docs_en)
sem_scores = semantic_search(query, docs_en, model_en)

print(f"{'문서':<38}{'키워드':<12}{'의미'}")
print("-" * 70)
for doc, k, s in zip(docs_en, kw_scores, sem_scores):
    print(f"{doc:<38}{k:<12.3f}{s:.4f}")
print("-" * 70)
print()

print("키워드 검색 1위:", docs_en[int(kw_scores.argmax())])
print("의미 검색 1위  :", docs_en[int(sem_scores.argmax())])
print()
print("'Feeding schedule for canines' 를 보자.")
print("  질문의 'feed', 'dog' 와 단어가 겹치지 않는다 (feeding≠feed, canines≠dog)")
print("  키워드 검색은 놓치지만 의미 검색은 찾아낸다.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4.5))

x = np.arange(len(docs_en))
width = 0.38

# 비교를 위해 두 점수를 0~1로 맞춘다
kw_norm = kw_scores / (kw_scores.max() + 1e-9)
sem_norm = (sem_scores - sem_scores.min()) / (sem_scores.max() - sem_scores.min() + 1e-9)

ax.barh(x - width/2, kw_norm, width, label="키워드 검색", color="#94A3B8")
ax.barh(x + width/2, sem_norm, width, label="의미 검색", color="#1E40AF")

ax.set_yticks(x)
ax.set_yticklabels([d[:34] for d in docs_en], fontsize=9)
ax.set_xlabel("점수 (상대값)")
ax.set_title(f"검색 방식 비교 — '{query}'")
ax.legend()
ax.grid(axis="x", alpha=0.3)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print("의미 검색이 관련 문서 전체를 고르게 찾아낸다.")
print()
print("다만 의미 검색이 항상 낫지는 않다.")
print("  - 고유명사·제품코드 등 정확히 일치해야 하는 경우 → 키워드가 유리")
print("  - 실무에서는 둘을 합친 하이브리드 검색을 쓰기도 한다 (이론편 21.4절).")

---

## 5. 모델 선택 — 영어 전용 vs 다국어

**지금까지 쓴 모델은 영어 전용이다.** 한국어를 넣으면 어떻게 될까.

이것이 실무에서 가장 자주 겪는 함정이다. 겉으로는 동작하는 것처럼 보이지만
결과가 엉뚱하게 나온다.

In [ ]:
import numpy as np

korean_sents = [
    "강아지 사료 추천",
    "반려견 먹이 고르는 법",
    "자동차 엔진오일 교체",
]

print("=" * 70)
print("영어 전용 모델에 한국어를 넣으면")
print("=" * 70)

emb = model_en.encode(korean_sents, normalize_embeddings=True)
S = emb @ emb.T

print(f"{'':24}" + "".join(f"{i:>10}" for i in range(3)))
print("-" * 70)
for i, s in enumerate(korean_sents):
    print(f"{s:<24}" + "".join(f"{S[i,j]:>10.3f}" for j in range(3)))
print("-" * 70)
for i, s in enumerate(korean_sents):
    print(f"  {i}: {s}")
print()

print("기대하는 결과")
print("  0 ↔ 1 (강아지 사료 ↔ 반려견 먹이): 높아야 함")
print("  0 ↔ 2 (강아지 사료 ↔ 자동차 정비): 낮아야 함")
print()
print("실제 결과")
print(f"  0 ↔ 1 = {S[0,1]:.3f}")
print(f"  0 ↔ 2 = {S[0,2]:.3f}")
print()
if S[0,2] >= S[0,1] - 0.05:
    print("[문제] 무관한 문서가 관련 문서만큼 높게 나왔다.")
else:
    print("[주의] 차이가 기대만큼 크지 않다.")
print()
print("원인: 23장 4절에서 본 대로 한국어가 잘게 쪼개진다.")
print("  의미를 담기 전에 토큰화 단계에서 이미 정보가 흐트러진다.")

In [ ]:
import time
from sentence_transformers import SentenceTransformer

MULTI_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print("=" * 60)
print("다국어 모델 불러오기")
print("=" * 60)
print(f"모델: {MULTI_MODEL}")
print("약 470MB를 내려받습니다 (처음 한 번).")
print()

t0 = time.time()
model_multi = SentenceTransformer(MULTI_MODEL)
print(f"로드 완료: {time.time()-t0:.1f}초")
print()

n_en = sum(p.numel() for p in model_en.parameters())
n_multi = sum(p.numel() for p in model_multi.parameters())

print(f"{'모델':<16}{'파라미터':<14}{'차원':<10}{'특징'}")
print("-" * 60)
print(f"{'영어 전용':<16}{n_en/1e6:<14.1f}{model_en.get_sentence_embedding_dimension():<10}빠름")
print(f"{'다국어':<16}{n_multi/1e6:<14.1f}{model_multi.get_sentence_embedding_dimension():<10}여러 언어 지원")
print("-" * 60)
print()
print("다국어 모델이 큰 이유: 여러 언어의 어휘를 모두 담아야 하므로")
print("  임베딩 층(이론편 20.3절)이 훨씬 커진다.")

In [ ]:
import numpy as np

print("=" * 70)
print("다국어 모델로 다시")
print("=" * 70)

emb_m = model_multi.encode(korean_sents, normalize_embeddings=True)
S_m = emb_m @ emb_m.T

print(f"{'':24}" + "".join(f"{i:>10}" for i in range(3)))
print("-" * 70)
for i, s in enumerate(korean_sents):
    print(f"{s:<24}" + "".join(f"{S_m[i,j]:>10.3f}" for j in range(3)))
print("-" * 70)
print()

print("두 모델 비교")
print(f"{'비교':<32}{'영어 전용':<14}{'다국어'}")
print("-" * 70)
print(f"{'강아지사료 ↔ 반려견먹이 (높아야)':<32}{S[0,1]:<14.3f}{S_m[0,1]:.3f}")
print(f"{'강아지사료 ↔ 자동차정비 (낮아야)':<32}{S[0,2]:<14.3f}{S_m[0,2]:.3f}")
print("-" * 70)
gap_en = S[0,1] - S[0,2]
gap_multi = S_m[0,1] - S_m[0,2]
print(f"{'관련/무관 격차':<32}{gap_en:<14.3f}{gap_multi:.3f}")
print()
if gap_multi > gap_en:
    print("[OK] 다국어 모델이 관련 문서와 무관 문서를 훨씬 잘 구분한다.")
print()
print("한국어를 다룬다면 반드시 다국어 또는 한국어 특화 모델을 써야 한다.")

In [ ]:
import numpy as np

print("=" * 70)
print("다국어 모델의 장점 — 언어를 넘나드는 검색")
print("=" * 70)

mixed = [
    "강아지 사료 추천",
    "dog food review",
    "자동차 엔진오일 교체",
    "car maintenance",
]

emb_mix = model_multi.encode(mixed, normalize_embeddings=True)
S_mix = emb_mix @ emb_mix.T

print(f"{'':24}" + "".join(f"{i:>10}" for i in range(4)))
print("-" * 70)
for i, s in enumerate(mixed):
    print(f"{s:<24}" + "".join(f"{S_mix[i,j]:>10.3f}" for j in range(4)))
print("-" * 70)
for i, s in enumerate(mixed):
    print(f"  {i}: {s}")
print()

print("주목할 점")
print(f"  '강아지 사료' ↔ 'dog food'    : {S_mix[0,1]:.3f}   ← 다른 언어, 같은 주제")
print(f"  '강아지 사료' ↔ '자동차 정비' : {S_mix[0,2]:.3f}   ← 같은 언어, 다른 주제")
print()
print("언어보다 **의미**가 더 중요하게 반영된다.")
print()
print("이것이 뜻하는 것")
print("  한국어로 질문해도 영어 문서를 찾을 수 있다.")
print("  다국어 문서를 하나의 저장소에서 함께 검색할 수 있다.")

---

## 6. 벡터 검색 구현 — 이론편 21.3절

이제 실제로 쓸 수 있는 검색기를 만든다.

**절차**
1. 모든 문서를 미리 벡터로 바꿔 저장 (인덱싱)
2. 질문이 오면 질문도 벡터로 바꿈
3. 모든 문서와 유사도 계산
4. 상위 k개 반환

In [ ]:
import numpy as np
import time


class SimpleVectorStore:
    # 벡터 저장소 (이론편 21.3절)
    #
    # 실무에서는 FAISS, Chroma 같은 전용 라이브러리를 쓰지만,
    # 원리는 이것과 같다. 28장에서 다룬다.

    def __init__(self, model):
        self.model = model
        self.documents = []
        self.embeddings = None

    def add(self, documents, batch_size=32, verbose=True):
        """문서를 추가하고 벡터로 변환해 저장한다"""
        t0 = time.time()
        new_emb = self.model.encode(
            documents, normalize_embeddings=True,
            batch_size=batch_size, show_progress_bar=False)

        if self.embeddings is None:
            self.embeddings = new_emb
        else:
            self.embeddings = np.vstack([self.embeddings, new_emb])
        self.documents.extend(documents)

        if verbose:
            print(f"  {len(documents)}개 추가 ({time.time()-t0:.2f}초) "
                  f"— 전체 {len(self.documents)}개")

    def search(self, query, top_k=3):
        """질문과 가장 가까운 문서 top_k개를 찾는다"""
        if self.embeddings is None:
            return []

        q_emb = self.model.encode([query], normalize_embeddings=True)[0]

        # 정규화되어 있으므로 내적 = 코사인 유사도 (3절)
        scores = self.embeddings @ q_emb

        top_idx = np.argsort(scores)[::-1][:top_k]
        return [(self.documents[i], float(scores[i])) for i in top_idx]


# 실습용 문서 모음
knowledge = [
    "파이썬은 배우기 쉬운 프로그래밍 언어로, 문법이 간결하고 읽기 쉽다.",
    "자바는 객체지향 언어로 기업용 애플리케이션 개발에 널리 쓰인다.",
    "머신러닝은 데이터로부터 규칙을 학습하는 방법론이다.",
    "딥러닝은 여러 층의 신경망을 사용하는 머신러닝의 한 갈래다.",
    "강아지에게는 하루 두 번 정해진 시간에 사료를 주는 것이 좋다.",
    "고양이는 물을 잘 마시지 않으므로 습식 사료를 함께 주면 도움이 된다.",
    "자동차 엔진오일은 주행거리 5000~10000km마다 교체를 권장한다.",
    "타이어 공기압은 월 1회 점검하는 것이 안전에 좋다.",
]

print("=" * 65)
print("벡터 저장소 구축")
print("=" * 65)
store = SimpleVectorStore(model_multi)
store.add(knowledge)
print()
print(f"저장된 벡터 모양: {store.embeddings.shape}")
print(f"메모리 사용     : {store.embeddings.nbytes/1024:.1f} KB")

In [ ]:
print("=" * 70)
print("검색 실행")
print("=" * 70)

queries = [
    "프로그래밍 언어 배우기",
    "반려동물 먹이 주기",
    "차량 정비 주기",
    "신경망이 뭐야?",
]

for q in queries:
    print(f"\n질문: {q}")
    print("-" * 70)
    for rank, (doc, score) in enumerate(store.search(q, top_k=2), 1):
        print(f"  {rank}. [{score:.4f}] {doc}")

print()
print("=" * 70)
print("겹치는 단어가 없어도 찾아낸다")
print("  '신경망이 뭐야?' → 딥러닝 문서 ('신경망'이라는 단어가 실제로 있음)")
print("  '반려동물 먹이 주기' → 강아지·고양이 문서 ('반려동물'은 없는 단어)")

---

## 7. 문서 분할(Chunking) — 이론편 21.2절

지금까지는 문서가 짧았다. **실제 문서는 길다.**

긴 문서를 통째로 벡터 하나에 담으면 두 가지 문제가 생긴다.

1. **모델의 입력 길이 제한** — 대부분 256~512 토큰
2. **의미가 희석됨** — 여러 주제가 섞여 평균 내면 무엇도 아닌 벡터가 됨

그래서 **적당한 크기로 잘라서(chunk) 저장**한다.

In [ ]:
import numpy as np

long_document = """파이썬은 1991년 귀도 반 로섬이 발표한 프로그래밍 언어다.
문법이 간결하고 읽기 쉬워 초보자가 배우기에 적합하다.
들여쓰기로 블록을 구분하는 것이 특징이다.

파이썬은 인터프리터 언어로, 코드를 한 줄씩 해석해 실행한다.
컴파일 과정이 없어 개발 속도가 빠르지만 실행 속도는 상대적으로 느리다.
이 때문에 성능이 중요한 부분은 C로 작성된 라이브러리를 사용한다.

데이터 분석과 인공지능 분야에서 파이썬이 널리 쓰이는 이유는 라이브러리 생태계 때문이다.
NumPy, Pandas, PyTorch 같은 도구들이 대부분 파이썬을 지원한다.
이들 라이브러리의 핵심 연산은 C나 CUDA로 구현되어 속도 문제를 해결한다."""

print("=" * 65)
print("긴 문서를 통째로 넣으면")
print("=" * 65)
print(f"문서 길이: {len(long_document)}자")

tokens = model_multi.tokenizer.encode(long_document)
print(f"토큰 수  : {len(tokens)}")
print(f"모델 한계: {model_multi.max_seq_length} 토큰")
print()

if len(tokens) > model_multi.max_seq_length:
    print(f"[문제] 한계를 {len(tokens)-model_multi.max_seq_length}토큰 초과한다.")
    print("  초과분은 **잘려서 버려진다** — 뒷부분 내용을 검색할 수 없게 된다.")
else:
    print("이 문서는 한계 안에 들어간다.")
print()
print("한계를 넘지 않더라도 문제가 있다:")
print("  세 문단이 각각 다른 주제인데(역사/실행방식/생태계),")
print("  하나의 벡터로 만들면 무엇을 물어도 어중간하게 매칭된다.")

In [ ]:
def split_by_paragraph(text):
    """문단 단위로 나누기 — 가장 단순한 방법"""
    return [p.strip() for p in text.split("\n\n") if p.strip()]


def split_by_length(text, chunk_size=100, overlap=20):
    """길이 기준으로 나누기 + 겹침(overlap)

    overlap 을 두는 이유:
      경계에서 문장이 잘리면 의미가 끊긴다.
      앞 조각의 끝부분을 다음 조각에 겹쳐 두면 문맥이 이어진다.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end].strip())
        start = end - overlap
        if start >= len(text) - overlap:
            break
    return [c for c in chunks if c]


print("=" * 70)
print("분할 방법 비교")
print("=" * 70)

para_chunks = split_by_paragraph(long_document)
len_chunks = split_by_length(long_document, chunk_size=120, overlap=30)

print(f"[문단 단위] {len(para_chunks)}개")
for i, c in enumerate(para_chunks):
    print(f"  {i}: ({len(c)}자) {c[:44]}...")
print()

print(f"[길이 기준 120자, 겹침 30자] {len(len_chunks)}개")
for i, c in enumerate(len_chunks[:4]):
    print(f"  {i}: ({len(c)}자) {c[:44]}...")
print()

print("-" * 70)
print("어느 쪽이 나은가")
print("  문단 단위: 의미가 온전히 보존됨. 단, 문단 길이가 제각각")
print("  길이 기준: 크기가 균일함. 단, 문장 중간에서 잘릴 수 있음")
print()
print("실무에서는 둘을 섞는다 —")
print("  문단으로 나눈 뒤, 너무 긴 것만 길이 기준으로 다시 자른다.")

In [ ]:
import numpy as np

print("=" * 70)
print("분할이 검색 품질에 미치는 영향")
print("=" * 70)

# 통째로 저장한 경우
store_whole = SimpleVectorStore(model_multi)
store_whole.add([long_document], verbose=False)

# 문단으로 나눠 저장한 경우
store_chunked = SimpleVectorStore(model_multi)
store_chunked.add(para_chunks, verbose=False)

test_queries = [
    "파이썬은 누가 만들었나",
    "파이썬이 느린 이유",
    "데이터 분석 라이브러리",
]

print(f"{'질문':<24}{'통째로':<12}{'문단 분할':<12}{'분할이 찾은 내용'}")
print("-" * 70)
for q in test_queries:
    r_whole = store_whole.search(q, top_k=1)[0]
    r_chunk = store_chunked.search(q, top_k=1)[0]
    snippet = r_chunk[0][:28].replace("\n", " ")
    print(f"{q:<24}{r_whole[1]:<12.4f}{r_chunk[1]:<12.4f}{snippet}...")

print("-" * 70)
print()
print("분할한 쪽의 점수가 대체로 높다.")
print("  질문과 관련된 부분만 담긴 조각이 더 정확히 매칭되기 때문이다.")
print()
print("더 중요한 것은 **무엇을 찾았는지**다.")
print("  통째로 저장하면 항상 같은 문서 하나만 나온다.")
print("  분할하면 질문에 맞는 부분을 정확히 짚어 준다.")
print("  → 23번 RAG에서 이 조각을 LLM에게 넘긴다.")

---

## 8. 검색 품질 평가 — 이론편 21.5절

**검색이 잘 되는지 어떻게 알 수 있을까.** 이론편 21.5절에서 다룬 지표를 계산한다.

| 지표 | 뜻 |
|---|---|
| Recall@k | 정답 문서가 상위 k개 안에 들어왔는가 |
| MRR | 정답이 몇 번째에 있었는가 (역순위 평균) |

07장에서 다룬 재현율과 같은 개념이지만, **순위가 있다**는 점이 다르다.

In [ ]:
import numpy as np

# 평가용 데이터 — 질문과 정답 문서의 짝
eval_data = [
    ("프로그래밍 언어 배우기", 0),      # knowledge[0] 파이썬
    ("객체지향 언어",          1),      # 자바
    ("데이터로 규칙 학습",      2),      # 머신러닝
    ("신경망 여러 층",         3),      # 딥러닝
    ("강아지 사료 주는 시간",   4),      # 강아지
    ("고양이 수분 섭취",       5),      # 고양이
    ("엔진오일 교체 주기",     6),      # 자동차
    ("타이어 점검",            7),      # 타이어
]


def evaluate(store, eval_data, k_values=(1, 3, 5)):
    """Recall@k 와 MRR 계산 (이론편 21.5절)"""
    hits = {k: 0 for k in k_values}
    reciprocal_ranks = []

    for query, correct_idx in eval_data:
        results = store.search(query, top_k=max(k_values))
        # 검색 결과에서 정답 문서의 순위 찾기
        ranked_docs = [doc for doc, _ in results]
        correct_doc = store.documents[correct_idx]

        if correct_doc in ranked_docs:
            rank = ranked_docs.index(correct_doc) + 1
            reciprocal_ranks.append(1.0 / rank)
            for k in k_values:
                if rank <= k:
                    hits[k] += 1
        else:
            reciprocal_ranks.append(0.0)

    n = len(eval_data)
    return ({k: hits[k]/n for k in k_values}, float(np.mean(reciprocal_ranks)))


print("=" * 65)
print("검색 품질 평가 (이론편 21.5절)")
print("=" * 65)
print(f"평가 질문 {len(eval_data)}개, 문서 {len(store.documents)}개")
print()

recalls, mrr = evaluate(store, eval_data)

print(f"{'지표':<16}{'값':<12}{'뜻'}")
print("-" * 65)
for k, v in recalls.items():
    print(f"{'Recall@'+str(k):<16}{v:<12.4f}상위 {k}개 안에 정답이 있는 비율")
print(f"{'MRR':<16}{mrr:<12.4f}정답의 평균 역순위")
print("-" * 65)
print()
print("MRR 읽는 법")
print("  1.0 = 항상 1위로 찾음")
print("  0.5 = 평균적으로 2위")
print("  0.33 = 평균적으로 3위")

In [ ]:
import numpy as np

print("=" * 70)
print("질문별 상세 — 어디서 틀렸나")
print("=" * 70)
print(f"{'질문':<24}{'정답 순위':<12}{'1위로 나온 문서'}")
print("-" * 70)

for query, correct_idx in eval_data:
    results = store.search(query, top_k=len(store.documents))
    ranked = [doc for doc, _ in results]
    correct_doc = store.documents[correct_idx]
    rank = ranked.index(correct_doc) + 1 if correct_doc in ranked else "없음"
    top1 = results[0][0][:32]
    mark = "" if rank == 1 else "  ←"
    print(f"{query:<24}{str(rank):<12}{top1}...{mark}")

print("-" * 70)
print()
print("1위가 아닌 경우를 보면 개선 방향이 보인다.")
print()
print("검색 품질을 높이는 방법 (이론편 21.4절)")
print("  1) 더 좋은 임베딩 모델 사용")
print("  2) 문서 분할 크기 조정")
print("  3) 키워드 검색과 결합 (하이브리드)")
print("  4) 재순위화(reranking) — 상위 후보를 다시 정밀 평가")

In [ ]:
import numpy as np

print("=" * 70)
print("모델에 따른 검색 품질 차이")
print("=" * 70)

# 영어 전용 모델로도 같은 평가
store_en = SimpleVectorStore(model_en)
store_en.add(knowledge, verbose=False)
recalls_en, mrr_en = evaluate(store_en, eval_data)

print(f"{'모델':<20}{'Recall@1':<14}{'Recall@3':<14}{'MRR'}")
print("-" * 70)
print(f"{'영어 전용':<20}{recalls_en[1]:<14.4f}{recalls_en[3]:<14.4f}{mrr_en:.4f}")
print(f"{'다국어':<20}{recalls[1]:<14.4f}{recalls[3]:<14.4f}{mrr:.4f}")
print("-" * 70)
print()
print("한국어 문서에서는 다국어 모델이 확실히 낫다.")
print("5절에서 확인한 차이가 검색 품질로 그대로 나타난다.")
print()
print("[교훈] 모델을 고르기 전에 **자기 데이터로 평가해 봐야 한다.**")
print("  벤치마크 순위가 높다고 내 데이터에서도 좋은 것은 아니다.")

---

## 9. 정리

### 확인한 이론편 값

| 이론편 절 | 내용 | 결과 |
|---|---|---|
| **21.3** | **코사인 1.000 / 0.995 / 0.070** | **일치** ✓ |
| 21.3 | 길이로 나누는 이유 | 실험 확인 ✓ |
| 21.2 | 키워드 검색의 한계 | 비교 확인 ✓ |
| 21.5 | Recall@k, MRR | 계산 ✓ |

### 기억할 것

| 항목 | 요점 |
|---|---|
| 코사인 유사도 | 길이 무관, 방향만 봄 |
| `normalize_embeddings=True` | 정규화하면 내적 = 코사인 |
| 벡터 검색 | 행렬 곱 한 번으로 전체 계산 |
| **모델 선택** | **한국어면 반드시 다국어 모델** |
| 다국어 모델 | 언어를 넘나드는 검색 가능 |
| Chunking | 의미 단위로 나눠야 정확히 매칭 |
| overlap | 경계에서 문맥이 끊기지 않게 |
| 평가 | **자기 데이터로 직접 측정** |

### 실무에서 자주 하는 실수

1. **영어 모델로 한국어 처리** — 5절에서 확인
2. **문서를 통째로 저장** — 7절에서 확인
3. **평가 없이 모델 선택** — 8절에서 확인
4. 정규화를 빼먹고 내적만 사용

### 다음 장

**28. RAG 파이프라인 구축** — 이 장에서 만든 검색기에 LLM을 붙인다.
이론편 25장의 전체 구조를 완성하고, **검색이 답변 품질에 어떤 영향을 주는지** 확인한다.

### 모델 선택의 영향을 그림으로

5절과 8절에서 확인한 차이를 나란히 놓고 본다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

fig, axes = plt.subplots(1, 3, figsize=(15, 4)) 

korean_sents_short = ["강아지 사료", "반려견 먹이", "자동차 정비"]

# --- 왼쪽: 영어 전용 모델 히트맵 ---
emb_en = model_en.encode(korean_sents_short, normalize_embeddings=True)
S_en = emb_en @ emb_en.T
ax = axes[0]
im = ax.imshow(S_en, cmap="Blues", vmin=0, vmax=1)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{S_en[i,j]:.2f}", ha="center", va="center",
                fontsize=10, color="white" if S_en[i,j] > 0.6 else "black")
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels(korean_sents_short, fontsize=8, rotation=20, ha="right")
ax.set_yticklabels(korean_sents_short, fontsize=8)
ax.set_title("영어 전용 모델")

# --- 가운데: 다국어 모델 히트맵 ---
emb_mu = model_multi.encode(korean_sents_short, normalize_embeddings=True)
S_mu = emb_mu @ emb_mu.T
ax = axes[1]
im = ax.imshow(S_mu, cmap="Blues", vmin=0, vmax=1)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{S_mu[i,j]:.2f}", ha="center", va="center",
                fontsize=10, color="white" if S_mu[i,j] > 0.6 else "black")
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels(korean_sents_short, fontsize=8, rotation=20, ha="right")
ax.set_yticklabels([], fontsize=8)
ax.set_title("다국어 모델")

# --- 오른쪽: 검색 성능 비교 ---
ax = axes[2]
metrics = ["Recall@1", "Recall@3", "MRR"]
en_vals = [recalls_en[1], recalls_en[3], mrr_en]
mu_vals = [recalls[1], recalls[3], mrr]
x = np.arange(len(metrics))
w = 0.36
ax.bar(x - w/2, en_vals, w, label="영어 전용", color="#DC2626")
ax.bar(x + w/2, mu_vals, w, label="다국어", color="#0D9488")
for i, (e, m) in enumerate(zip(en_vals, mu_vals)):
    ax.text(i - w/2, e + 0.02, f"{e:.2f}", ha="center", fontsize=8)
    ax.text(i + w/2, m + 0.02, f"{m:.2f}", ha="center", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(metrics, fontsize=9)
ax.set_ylabel("점수")
ax.set_ylim(0, 1.18)
ax.set_title("한국어 검색 성능")
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

gap_en = S_en[0,1] - S_en[0,2]
gap_mu = S_mu[0,1] - S_mu[0,2]
print("왼쪽·가운데: 관련 문장(0-1)과 무관 문장(0-2)의 격차")
print(f"  영어 전용 {gap_en:+.3f}  vs  다국어 {gap_mu:+.3f}")
print()
print("오른쪽: 그 차이가 검색 성능으로 그대로 나타난다")
print("  한국어를 다룬다면 모델 선택이 다른 어떤 개선보다 중요하다.")